## 5.2 랭그래프 기본 개념 이해하고 적용하기

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

### 5.2.1 [실습] 그래프의 상태 정의하기

In [2]:
from typing_extensions import TypedDict

class State(TypedDict):
    messages: list[str]

In [3]:
from typing import TypedDict

class User(TypedDict):
    id: int
    name: str
    email: str

user1: User = {
    'id': 1,
    'name': 'nayeon_park',
    'email': 'example@gmail.com'
}
print(user1)

{'id': 1, 'name': 'nayeon_park', 'email': 'example@gmail.com'}


In [4]:
user1: User = {
    'id': 1,
    'name': 123,
    'email': 'example@gmail.com'
}
print(user1)

{'id': 1, 'name': 123, 'email': 'example@gmail.com'}


In [5]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    name: str
    email: str

user_data = {
    'id': 1,
    'name': 'nayeon_park',
    'email': 'example@gmail.com'
}

user1 = User(**user_data)
print(user1)

id=1 name='nayeon_park' email='example@gmail.com'


In [6]:
user_data = {
    'id': 1,
    'name': 123,
    'email': 'example@gmail.com'
}

user = User(**user_data)
print(user)

ValidationError: 1 validation error for User
name
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type

### 5.2.2 [실습] 그래프의 상태에 리듀서 함수 추가하기

In [7]:
from typing import TypedDict, Annotated

def add(left, right):
    return left + right

class State(TypedDict):
    messages: Annotated[list[str], add]

In [8]:
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph.message import add_messages

msgs1 = [HumanMessage(content="Hello", id="1")]
msgs2 = [AIMessage(content="Hi there!", id="2")]

add_messages(msgs1, msgs2)

[HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='1'),
 AIMessage(content='Hi there!', additional_kwargs={}, response_metadata={}, id='2')]

In [9]:
msgs1 = [HumanMessage(content="Hello", id="1")]
msgs2 = [HumanMessage(content="Hello again", id="1")]

add_messages(msgs1, msgs2)

[HumanMessage(content='Hello again', additional_kwargs={}, response_metadata={}, id='1')]

In [10]:
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

### 5.2.3 [실습] 그래프의 실행 단위, 노드 추가하기

In [11]:
from typing import TypedDict, Annotated
from operator import add

from langgraph.graph import StateGraph

class State(TypedDict):
    messages: Annotated[list[str], add]

graph = StateGraph(State)

def chatbot(state: State): # [ 1 ]
    question = state["messages"]
    answer = f"사용자 입력을 그대로 반환하는 챗봇입니다. {question} 라는 질문을 받았습니다." # [ 2 ]
    return {"messages": [answer]} # [ 3 ]

graph.add_node("chatbot", chatbot) # [ 4 ]

### 5.2.5 [실습] 그래프에 조건부 엣지 추가하기

In [12]:
from langgraph.graph import START, END

graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

In [13]:
graph.add_edge("node_a", "node_b")

**조건부 엣지**

In [ ]:
def routing_function(state: State):
    if len(state["messages"][-1]) > 1000 :
        return True
    return False

graph.add_conditional_edges(
	"chatbot",
	routing_function,
	{True: "summary", False: END}
)